# 疾患メカニズムに関わる遺伝子を順位づける（guidance + transformers 版）

`gene_disease_ranking03.ipynb` と同じ処理を、**Ollama ではなく guidance と transformers で**動かすものです。
Hugging Face のモデル（`epfl-llm/meditron-70b` など）を transformers で読み込み、guidance で問いかけます。
段階の流れ・プロンプト・保存の形は 03 と同じなので、結果をそのまま比べられます。
関数（`def`）は、何度も使う 2 つ（モデルに聞く・Bradley-Terry で強さを出す）だけです。

## guidance の使い方（このノートブックで使う形）

```python
llm = models.Transformers(model, tokenizer, echo=True, top_k=20)   # transformers で読み込んだモデルを guidance に渡す
llm_state = llm.copy() + prompt                                    # 問いを足した「状態」を作る（元の llm は変わらない）
llm_state += select([" Yes", " No"], name="answer")                # 答えを選択肢の中の 1 つに限って、続きを出させる
```

- guidance のモデルは**書き換わらない**オブジェクトで、`+` のたびに新しい状態ができます。
  同じ `llm` から何千回問いかけても、前の問いの影響は残りません。
- `select` で答えを選択肢に限るので、**モデルが遺伝子記号を自由に書くことはありません**（記号は複数トークンに分かれ、
  書かせると GPR52 と GPR56 のような似た記号を取り違えるため）。選択肢の文字と遺伝子の対応はプログラムが持ちます。
- 読むのは「選んだ答え」ではなく、**制約をかける前の、次のトークンの確率**です（段階0 は ` Yes` / ` No`、選択式は ` A`〜` F`）。

## 必要なもの

```
pip install guidance transformers torch accelerate
```

CUDA の GPU で 4 ビット読み込み（`LOAD_IN_4BIT`）を使うときは `bitsandbytes` も入れます。
meditron-70b は利用条件への同意が必要なリポジトリなので、Hugging Face のモデルのページで同意し、
読み取り用トークンを環境変数 `HF_TOKEN` に入れるか `huggingface-cli login` でログインしておきます（トークンはノートブックに書かない）。

## 全体の流れ

| 段階 | やること | 既定の規模 | モデルに聞く回数 |
|---|---|---|---|
| 0 | 1 遺伝子ずつ「メカニズムのどれかの段階を変えるか」を Yes / No で聞く | 全候補 → 2,000 | 候補数 |
| 0.5 | 5 遺伝子＋「その他」から 1 つ選ばせる。全員 4 回ずつ出題し、粗く並べる | 2,000 → 500 | 1,600 |
| 1 | 同じ選択式を全員 16 回ずつ出題し、詳しく並べる | 500 → 100 | 1,600 |
| 2 | 2 遺伝子＋「どちらも関係ない」の総当たり（A/B を入れ替えて 2 回） | 100 → 順位 | 9,900 |

## 結果を読む前に（これまでの測定で分かったこと）

- **上位の顔ぶれはメカニズムの書き方で大きく変わります。** 統合失調症で記述を替えると C4A が 0.96 → 0.00、
  DRD2 が 0.27 → 0.89 と動きました（qwen3:14b と gemma3:27b で同じ傾向）。結果は「この記述に合う遺伝子」として読んでください。
- **モデルが知らないつながりは拾えません。** 承認薬の標的 CHRM4 は、聞き方をどう変えても下位のままでした。
  下位にあることは「関係ない」の証拠になりません。meditron-70b でどうなるかは未測定です。
- 詳しくは `EXPERIMENTS.md` の「ノートブック 02」の節にあります。

## 1. 入力 — 疾患・メカニズム・候補遺伝子

**解析ごとに変えるのはこのセルだけです。**

`DISEASE_MECHANISM` の各段階は「**変えられる要因**」を主語にして書きます。
たとえばシスチン尿症では「腎臓が尿を濃縮するほど、尿中のシスチン濃度が上がる」と書くと、
尿を濃くするホルモンの受容体 AVPR2 に届きました。「シスチンが尿中で濃縮して結石になる」では届きませんでした。

遺伝子名・受容体名・薬の名前は書きません。書くと答えを教えることになります。

候補遺伝子のファイルは、1 行 1 遺伝子のタブ区切りです:
`記号<TAB>HGNC 正式名<TAB>UniProt の蛋白質名<TAB>別名（カンマ区切り）`（2〜4 列目は無くても動きます）。
`KNOWN_ANSWERS` は答え合わせ用で、空でも構いません。

In [ ]:
DISEASE = "schizophrenia"

DISEASE_MECHANISM = [
    "The more synapses are eliminated in the cortex during adolescence, the fewer cortical synapses remain.",
    "The weaker glutamate signaling onto cortical inhibitory neurons, the less synchronized cortical activity becomes and the worse working memory gets.",
    "The less the prefrontal cortex restrains midbrain dopamine neurons, the more dopamine the striatum releases.",
    "The more dopamine the striatum releases, the stronger the hallucinations and delusions.",
]

GENE_FILE = "genelist_schizophrenia_mech1000.txt"

KNOWN_ANSWERS = ["C4A", "GRIN2A", "GRM3", "DRD2", "DRD3", "CHRM4", "COMT", "GAD1", "SLC6A3"]

## 2. 設定

ふだんは変えなくて構いません。数字はこれまでの測定で決めたものです。

- **モデル（`MODEL_PATH`）**: Hugging Face の名前（`epfl-llm/meditron-70b`）か、手元のフォルダを書きます。
  初回は Hugging Face からダウンロードし、2 回目からはキャッシュを使います。

  **大きさに注意してください。** meditron-70b は 16 ビットで 138 GB あり、読み込むにはそれだけの GPU メモリが要ります。
  CUDA の GPU なら `LOAD_IN_4BIT = True` で約 40 GB に縮められます（`bitsandbytes` が必要）。
  このマシン（Apple M4・メモリ 25.8 GB）には載らないので、読み込む前に止めます。動作確認には小さいモデル
  （`Qwen/Qwen2.5-0.5B-Instruct` など）を使ってください。
  meditron は会話用ではない素のモデル（Llama 2 系、文脈 4,096 トークン）ですが、このノートブックは例題を並べて
  続きの 1 文字を読む形なので、そのまま使えます。この課題での精度は未測定です。
- **置き場所（`DEVICE`）と精度（`DTYPE`）**: `"auto"` なら CUDA → MPS（Mac の GPU）→ CPU の順に使えるものを選び、
  CUDA は bfloat16、MPS は float16、CPU は float32 で読み込みます。
- **残す数と登場回数**: 段階0.5 は「粗く並べて真の上位を落とさない」役目なので 4 回で足ります
  （シミュレーションで、真の上位 20 の 100% が 500 個に残った）。段階1 は 16 回で、真の上位 20 が 100 個に残りました。
- **質問文**: 段階0 と選択式で文の形が違います（Yes/No で答える形と、どれかを選ぶ形）。
- **見本（few-shot）**: 本番と同じ形の例題を先に見せます。題材は嚢胞性線維症と関節リウマチで、
  対象の疾患とは別にしてあります。段階0 の見本で DNASE1（原因ではないが、粘液を薄くする）を Yes にしているのは、
  「原因遺伝子だけが Yes」ではないことを示すためです。

In [ ]:
# ===== モデル =====
MODEL_PATH = "epfl-llm/meditron-70b"   # Hugging Face の名前か、手元のフォルダ。動作確認には "Qwen/Qwen2.5-0.5B-Instruct" など
DEVICE = "auto"           # "auto" / "cuda" / "mps" / "cpu"
DTYPE = "auto"            # "auto" / "bfloat16" / "float16" / "float32"
LOAD_IN_4BIT = False      # CUDA だけ。bitsandbytes で 4 ビットに読み込む（meditron-70b が 138 GB → 約 40 GB）
MODEL_DOWNLOAD_DIR = ""   # ダウンロード先。空なら Hugging Face の既定（~/.cache/huggingface）
TOP_K = 20                # 次のトークンの候補を、確率つきで何個まで受け取るか

# ===== 各段階で残す数・1 遺伝子あたりの登場回数 =====
STAGE0_KEEP = 2000        # 段階0 から段階0.5 へ送る数（候補数より多ければ全員送る）
GROUP_SIZE = 5            # 選択式の 1 問に並べる遺伝子の数（「その他」を除く）
CHOICE_ROUNDS = [         # (名前, 残す数, 1 遺伝子の登場回数, 乱数の種)
    ("段階0.5", 500, 4, 0),
    ("段階1", 100, 16, 1),
]
BOTH_WAYS = True          # 段階2 で A/B を入れ替えて 2 回聞く（選択肢の順番の偏りを打ち消す）

# ===== 質問文 =====
BINARY_QUESTION = "Would changing this gene's activity, in either direction, change any of these steps of the disease mechanism?"
CHOICE_QUESTION = "Changing which gene's activity, in either direction, would change any of these steps of the disease mechanism?"
CHOICE_EXIT = "Other / none of the above"   # 段階0.5・1 の「その他」
PAIR_EXIT = "Neither of these genes"        # 段階2 の第3の選択肢

# ===== 選択肢の表示（段階0.5 以降）=====
# 記号 (蛋白質名) (aliases: 別名) の形。段階0 は記号だけ（蛋白質名を足すと正解の順位がかえって下がった）
MAX_ALIASES = 4

# ===== Bradley-Terry =====
ALPHA = 0.5               # 仮想の引き分け。一度も勝てない遺伝子の強さが -∞ に飛ぶのを防ぐ
NONE = "(none of these)"  # 「その他」「どちらも関係ない」を表す仮想の対戦相手

# ===== 見本（few-shot）=====
FEWSHOT_MECHANISM = {
    "Cystic fibrosis": [
        "A defective chloride channel dehydrates the airway surface.",
        "Extracellular DNA released from dead immune cells makes airway mucus thick.",
    ],
    "Rheumatoid arthritis": [
        "Immune cells release inflammatory cytokines that inflame the joint lining.",
        "The inflamed joint lining erodes cartilage and bone.",
    ],
}
BINARY_EXAMPLES = [("CFTR", "Yes"), ("APOE", "No"), ("DNASE1", "Yes"), ("HBB", "No")]   # 段階0（嚢胞性線維症）
CHOICE_EXAMPLES = [   # 段階0.5・1。正解の文字を B と C に分ける（同じ文字に寄せると、その文字を選ぶ癖がつく）
    ("Cystic fibrosis", ["HBB", "CFTR", "GPR52", "APOE"], "CFTR"),
    ("Rheumatoid arthritis", ["CFTR", "HBB", "TNF", "GPR56"], "TNF"),
]
PAIR_EXAMPLES = [     # 段階2。正解は B と A に分ける
    ("Cystic fibrosis", ["HBB", "CFTR"], "CFTR"),
    ("Rheumatoid arthritis", ["TNF", "GPR56"], "TNF"),
]
FEWSHOT_PROTEIN = {
    "HBB": "Hemoglobin subunit beta", "CFTR": "Cystic fibrosis transmembrane conductance regulator",
    "GPR52": "G-protein coupled receptor 52", "APOE": "Apolipoprotein E", "TNF": "Tumor necrosis factor",
    "GPR56": "Adhesion G-protein coupled receptor G1", "DNASE1": "Deoxyribonuclease-1",
}
FEWSHOT_ALIASES = {"CFTR": ["ABCC7"], "TNF": ["TNFA", "TNFSF2"], "GPR56": ["ADGRG1", "TM7LN4", "TM7XN1"], "DNASE1": ["DNL1", "DRNI"]}

# ===== 保存先 =====
OUTPUT_ROOT = "outputs"
RESUME_DIR = ""           # 途中から再開するときに、前回の実行フォルダ（例 "outputs/20260919-101500-nb04"）を入れる

## 3. ライブラリと保存先

- guidance・transformers・torch を読み込みます。
- 実行ごとに `outputs/日付-時刻-nb04/` を作り、段階ごとの結果をそこに保存します。
  `RESUME_DIR` を指定すると、そのフォルダにある段階は読み込むだけでモデルを呼びません（モデルの読み込み自体は行います）。

In [ ]:
import json, math, os, random, shutil, statistics, time
from collections import defaultdict

import torch
import huggingface_hub
from transformers import AutoModelForCausalLM, AutoTokenizer
from guidance import models, select

if RESUME_DIR:
    RUN_DIR = RESUME_DIR
    print("再開:", RUN_DIR, sorted(os.listdir(RUN_DIR)))
else:
    RUN_DIR = os.path.join(OUTPUT_ROOT, time.strftime("%Y%m%d-%H%M%S") + "-nb04")
    suffix = 2
    while os.path.exists(RUN_DIR):                       # 同じ秒に 2 回始めたとき
        RUN_DIR = os.path.join(OUTPUT_ROOT, time.strftime("%Y%m%d-%H%M%S") + f"-nb04-{suffix}")
        suffix += 1
    os.makedirs(RUN_DIR)
    print("保存先:", RUN_DIR)
with open(os.path.join(RUN_DIR, "input.json"), "w") as f:     # この実行の入力を残す
    json.dump({"disease": DISEASE, "mechanism": DISEASE_MECHANISM, "gene_file": GENE_FILE, "model_path": MODEL_PATH,
               "device": DEVICE, "dtype": DTYPE, "load_in_4bit": LOAD_IN_4BIT,
               "stage0_keep": STAGE0_KEEP, "rounds": CHOICE_ROUNDS}, f, ensure_ascii=False, indent=1)
print("guidance / transformers / torch:", __import__("guidance").__version__, __import__("transformers").__version__, torch.__version__)

## 4. モデルを読み込む（transformers → guidance）

1. **置き場所と精度を決めます**（`DEVICE` / `DTYPE` が `"auto"` のとき）。
2. **読み込む前に大きさを確かめます。** Hugging Face は目録だけを取って重みの大きさを調べ、
   必要なメモリ（16 ビットならほぼ重みの大きさ、4 ビットなら約 3 割、float32 なら 2 倍）が
   使えるメモリ（CUDA は GPU の合計、MPS と CPU はマシンのメモリ）の 9 割を超えるなら止めます。
   まだダウンロードしていなければ、ディスクの空きも確かめます。
3. **利用条件への同意が必要なリポジトリ**なら、トークンがあるかを確かめます（`HF_TOKEN` か `huggingface-cli login`）。
4. **transformers で読み込みます。** CUDA は `device_map="auto"` で複数の GPU に分けて載せます。
   MPS は読み込んでから `.to("mps")` で移します（`device_map="mps"` は guidance と組み合わせると止まったため）。
5. **guidance に渡します。** `echo=True` にすると、生成したトークンごとに**制約をかける前の確率**（上位 `TOP_K` 個）が
   記録されます。`echo` は画面表示も兼ねているので、作ったあとに `llm.echo = False` で表示だけ止めます
   （確率の記録はモデルを作ったときに決まり、表示の有無は問いかけのたびに `echo` を見て決まる）。

In [ ]:
# ===== 1. 置き場所と精度 =====
device = DEVICE if DEVICE != "auto" else ("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
dtype_name = DTYPE if DTYPE != "auto" else {"cuda": "bfloat16", "mps": "float16", "cpu": "float32"}[device]
dtype = getattr(torch, dtype_name)
if LOAD_IN_4BIT and device != "cuda":
    raise ValueError("LOAD_IN_4BIT は CUDA の GPU でだけ使えます")
print(f"置き場所 {device} / 精度 {dtype_name}" + (" / 4 ビット" if LOAD_IN_4BIT else ""))

# ===== 2. 大きさとメモリ・ディスク =====
token = os.environ.get("HF_TOKEN") or huggingface_hub.get_token()      # 画面には出さない
is_local = os.path.isdir(MODEL_PATH)
if is_local:
    names = os.listdir(MODEL_PATH)
    weight_files = [n for n in names if n.endswith(".safetensors")] or [n for n in names if n.endswith(".bin")]
    weights_gb = sum(os.path.getsize(os.path.join(MODEL_PATH, n)) for n in weight_files) / 1e9
    gated = False
else:
    info = huggingface_hub.HfApi().model_info(MODEL_PATH, files_metadata=True, token=token)
    sizes = {s.rfilename: s.size or 0 for s in info.siblings}
    weight_files = [n for n in sizes if n.endswith(".safetensors")] or [n for n in sizes if n.endswith(".bin")]
    weights_gb = sum(sizes[n] for n in weight_files) / 1e9
    gated = bool(info.gated)
need_gb = weights_gb * (0.3 if LOAD_IN_4BIT else 2.0 if dtype_name == "float32" else 1.0)
if device == "cuda":
    have_gb = sum(torch.cuda.mem_get_info(i)[1] for i in range(torch.cuda.device_count())) / 1e9
    where = f"GPU {torch.cuda.device_count()} 枚の合計"
else:
    have_gb = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9
    where = "このマシンのメモリ"
print(f"{MODEL_PATH}: 重み {weights_gb:.1f} GB → 読み込みに約 {need_gb:.1f} GB / {where} {have_gb:.1f} GB")
if need_gb > have_gb * 0.9:
    raise MemoryError(f"約 {need_gb:.1f} GB のモデルは {where}（{have_gb:.1f} GB）に載りません。"
                      "CUDA なら LOAD_IN_4BIT = True を試すか、メモリの大きいマシンで動かしてください")

if not is_local:
    cached = huggingface_hub.try_to_load_from_cache(MODEL_PATH, weight_files[-1], cache_dir=MODEL_DOWNLOAD_DIR or None)
    if not isinstance(cached, str):                      # まだダウンロードしていない
        cache_dir = MODEL_DOWNLOAD_DIR or os.path.expanduser("~/.cache/huggingface")
        os.makedirs(cache_dir, exist_ok=True)
        free_gb = shutil.disk_usage(cache_dir).free / 1e9
        print(f"ダウンロード {weights_gb:.1f} GB / 空きディスク {free_gb:.0f} GB（{cache_dir}）")
        if weights_gb > free_gb:
            raise OSError(f"ディスクが足りません。MODEL_DOWNLOAD_DIR を空きの大きいディスクにしてください")

# ===== 3. 利用条件への同意が必要なリポジトリ =====
if gated and not token:
    raise PermissionError(f"{MODEL_PATH} は利用条件への同意が必要です。https://huggingface.co/{MODEL_PATH} で同意し、"
                          "読み取り用トークンを環境変数 HF_TOKEN に入れるか huggingface-cli login でログインしてください")

# ===== 4. transformers で読み込む =====
start = time.time()
load_args = {"dtype": dtype, "token": token, "cache_dir": MODEL_DOWNLOAD_DIR or None}
if device == "cuda":
    load_args["device_map"] = "auto"                     # 複数の GPU に分けて載せる
if LOAD_IN_4BIT:
    from transformers import BitsAndBytesConfig
    load_args["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=dtype)
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, token=token, cache_dir=MODEL_DOWNLOAD_DIR or None)
model = AutoModelForCausalLM.from_pretrained(MODEL_PATH, **load_args)
if device != "cuda":
    model = model.to(device)                             # MPS は device_map ではなく、読み込んでから移す
model.eval()

# ===== 5. guidance に渡す =====
llm = models.Transformers(model, tokenizer, echo=True, top_k=TOP_K)   # echo=True で、トークンごとの確率を記録させる
llm.echo = False                                                    # 画面表示だけ止める（確率の記録は続く）
print(f"読み込み完了 {time.time() - start:.0f} 秒  パラメータ {sum(p.numel() for p in model.parameters()) / 1e9:.1f} B  置き場所 {model.device}")

## 5. 候補遺伝子を読む

ファイルから、遺伝子記号の並び（`GENES`）と、選択式で見せる表示（`OPTION_TEXT`）を作ります。

表示は `記号 (蛋白質名) (aliases: 別名)` です。蛋白質名は、同じ系統の遺伝子（SLC7A9 と SLC7A11 など）を
見分けるのに効きました。別名は、よく知られた呼び名（SLC5A2 = SGLT2 など）で取り違えないために付けます。
ただし別名のうち 590 個は別の遺伝子の正式記号と同じ文字列なので、紛らわしいことがあります（AOC1 の別名 DAO など）。

In [ ]:
GENES, PROTEIN, ALIASES, seen = [], {}, {}, set()
with open(GENE_FILE) as f:
    for line in f:
        if not line.strip() or line.startswith("#"):
            continue
        cols = (line.rstrip("\n").split("\t") + ["", "", ""])[:4]
        symbol = cols[0].strip()
        if not symbol or symbol in seen:
            continue                                   # 重複は 1 回だけ数える
        seen.add(symbol)
        GENES.append(symbol)
        if cols[2].strip():
            PROTEIN[symbol] = cols[2].strip()
        if cols[3].strip():
            ALIASES[symbol] = [a.strip() for a in cols[3].split(",") if a.strip()]

# 選択式で見せる 1 行分の表示を、全員分まとめて作っておく（見本の遺伝子も含める）
OPTION_TEXT = {}
for symbol in GENES + list(FEWSHOT_PROTEIN):
    protein = PROTEIN.get(symbol) or FEWSHOT_PROTEIN.get(symbol)
    aliases = (ALIASES.get(symbol) or FEWSHOT_ALIASES.get(symbol) or [])[:MAX_ALIASES]
    text = symbol + (f" ({protein})" if protein else "")
    if aliases:
        text += f" (aliases: {', '.join(aliases)})"
    OPTION_TEXT[symbol] = text

print(f"候補 {len(GENES)} 遺伝子（{GENE_FILE}）  蛋白質名あり {len(PROTEIN)}  別名あり {len(ALIASES)}")
print("例:", OPTION_TEXT[GENES[0]])
missing = [g for g in KNOWN_ANSWERS if g not in GENES]
print("答え合わせ:", KNOWN_ANSWERS, f"⚠ 候補に無い: {missing}" if missing else "（すべて候補に含まれる）")

## 6. 道具 1 — guidance で問いかけ、選択肢ごとの確率を読む

どの段階でも使うので、ここだけ関数にします。やることは次の 3 つです。

1. `llm.copy() + prompt` で、問いを足した状態を作ります。
2. `+= select(options, name="answer")` で、答えを選択肢のどれか 1 つに限って続きを出させます。
3. そのとき生成されたトークンの記録（`_trace_nodes`）から、**制約をかける前の、次のトークンの上位 `TOP_K` 個の確率**を取り出し、
   選択肢ごとの log 確率 `{"Yes": -0.13, "No": -2.17}` の形にして返します。上位に入らなかった選択肢は入れません。

`lm.log_prob("answer")` という関数もありますが、制約の中での確率なので、どの答えでも 0 になり使えません。

確かめたこと（Qwen2.5-0.5B-Instruct・Mac の MPS）: 段階0 のプロンプトで、ここで読んだ log P(Yes) − log P(No) が
transformers で直接計算した値と**小数第 4 位まで一致**しました。300 遺伝子で 0.07 秒/件、メモリは増えませんでした。

`_trace_nodes` は guidance の内部の属性なので、guidance の版が変わると動かなくなることがあります（0.3.1 で確認）。

In [ ]:
def option_logprobs(prompt, options):
    state = llm.copy() + prompt                           # 問いを足した状態（元の llm は変わらない）
    state += select(options, name="answer")               # 答えを選択肢の 1 つに限る
    generated = [o for node in state._trace_nodes for o in node.output
                 if type(o).__name__ == "TokenOutput" and not o.is_input]
    candidates = generated[0].top_k                       # 制約をかける前の、次のトークンの上位 TOP_K 個
    result = {}
    for option in options:
        probs = ([c.prob for c in candidates if c.token == option]                          # " Yes" そのもの
                 or [c.prob for c in candidates if c.token.strip() == option.strip()])     # 空白の付き方が違うトークン
        if probs and probs[0] > 0:
            result[option.strip()] = math.log(probs[0])
    return result


# 試しに 1 回聞いてみる
test = option_logprobs("Answer Yes or No.\nIs CFTR the gene mutated in cystic fibrosis?\nAnswer:", [" Yes", " No"])
print({k: round(v, 3) for k, v in test.items()})

## 7. 道具 2 — Bradley-Terry で「強さ」を出す

選択式の出題は、1 問ごとに「並んだ遺伝子それぞれが選ばれた確率」が得られます。
これを全部まとめて、**遺伝子ごとの強さ**を 1 つの数にします（Bradley-Terry / Luce のモデル）。

- 強い相手ばかりの群で勝った遺伝子は、弱い相手ばかりの群で勝った遺伝子より高く評価されます
  （平均確率で比べると、弱い群に当たり続けた遺伝子が得をします）。
- 「その他」「どちらも関係ない」も仮想の遺伝子 `NONE` として一緒に強さを出します。
  **NONE より弱い遺伝子は、「関係ない」に負けることが多かった遺伝子**です。
- 計算は MM 法の繰り返し（最尤推定）です。`ALPHA` は、強さ 1 の仮想の相手との引き分けを足す補正です。

返り値は `{遺伝子: log 強さ}` です。見やすさのため、表示では `elo = 1500 + 400 × log 強さ / log 10` に直します。

In [ ]:
def bradley_terry(matches, iterations=500, tolerance=1e-9):
    wins, groups_of = defaultdict(float), defaultdict(list)
    for group, probs in matches:                  # 勝ち数は確率の合計（小数で数える）
        for g in group:
            wins[g] += probs.get(g, 0.0)
            groups_of[g].append(group)
    strength = {g: 1.0 for g in wins}
    for _ in range(iterations):
        group_sum, new = {}, {}
        for g in strength:
            denom = 0.0
            for group in groups_of[g]:
                s = group_sum.get(id(group))
                if s is None:
                    s = group_sum[id(group)] = sum(strength[x] for x in group)
                denom += 1.0 / s
            denom += 2 * ALPHA / (strength[g] + 1.0)
            new[g] = (wins[g] + ALPHA) / denom
        scale = math.exp(statistics.mean(math.log(v) for v in new.values()))   # 全体の大きさをそろえる
        new = {g: v / scale for g, v in new.items()}
        change = max(abs(new[g] - strength[g]) for g in strength)
        strength = new
        if change < tolerance:
            break
    return {g: math.log(v) for g, v in strength.items()}


to_elo = lambda log_strength: 1500 + 400 * log_strength / math.log(10)
LETTERS = list("ABCDEFGHIJKLMNOPQRSTUVWXYZ")

## 8. 段階0 — 全候補を 1 遺伝子ずつ Yes / No で聞く

プロンプトは次の形です（見本 4 問のあとに、本番の 1 問）。

```
Answer Yes or No.

Disease: Cystic fibrosis
Disease mechanism:
- A defective chloride channel dehydrates the airway surface.
- ...
Would changing this gene's activity, in either direction, change any of these steps of the disease mechanism?
Gene: CFTR
Answer: Yes
  （APOE → No、DNASE1 → Yes、HBB → No の見本が続く）

Disease: <対象の疾患>
Disease mechanism:
- <段階 1>
- ...
Would changing ... ?
Gene: <記号>
Answer:
```

- **遺伝子記号をいちばん最後に置きます。** それより前は全遺伝子で共通なので、transformers が計算済みの部分（KV キャッシュ）を使い回します（Qwen2.5-0.5B・Mac の MPS で 1 遺伝子 0.07 秒。70B は GPU しだい）。
- 点数は **log P(Yes) − log P(No)** です。正なら Yes 寄り、負なら No 寄り。
- 点数の高い順に `STAGE0_KEEP` 個を段階0.5 へ送ります。ここは「正解を落とさない」ための緩いふるいで、残した中の順番は問いません。
- 分かっている弱点: 上位は +25〜+30 に大勢が並び、差がつきません（飽和）。蛋白質名を足すと飽和は緩みますが、正解の順位はかえって下がりました。
- 結果は `stage0.tsv` に保存します。あれば読み込むだけです。

In [ ]:
mechanism_block = f"Disease: {DISEASE}\nDisease mechanism:\n" + "".join(f"- {s}\n" for s in DISEASE_MECHANISM)

# 見本 4 問 ＋ 本番の問いの「Gene: 」まで（全遺伝子で共通の部分）
binary_prefix = "Answer Yes or No.\n\n"
for gene, answer in BINARY_EXAMPLES:
    binary_prefix += ("Disease: Cystic fibrosis\nDisease mechanism:\n"
                      + "".join(f"- {s}\n" for s in FEWSHOT_MECHANISM["Cystic fibrosis"])
                      + f"{BINARY_QUESTION}\nGene: {gene}\nAnswer: {answer}\n\n")
binary_prefix += mechanism_block + f"{BINARY_QUESTION}\nGene: "

stage0_file = os.path.join(RUN_DIR, "stage0.tsv")
if os.path.exists(stage0_file):
    STAGE0_SCORE = {}
    with open(stage0_file) as f:
        next(f)
        for line in f:
            rank, gene, score = line.rstrip("\n").split("\t")
            STAGE0_SCORE[gene] = float(score)
    print(f"段階0 は保存済みを読み込みました（{len(STAGE0_SCORE)} 遺伝子）")
else:
    STAGE0_SCORE = {}
    start = time.time()
    for i, gene in enumerate(GENES, 1):
        logprobs = option_logprobs(binary_prefix + gene + "\nAnswer:", [" Yes", " No"])
        STAGE0_SCORE[gene] = logprobs.get("Yes", -99.0) - logprobs.get("No", -99.0)   # 上位 TOP_K に無い側は -99
        if i == 10 or i % 500 == 0 or i == len(GENES):      # 最初の 10 件で全体の見積もりを出す（大きいモデル向け）
            elapsed = time.time() - start
            print(f"  {i}/{len(GENES)}  経過 {elapsed/60:.1f} 分  残り {(len(GENES)-i)*elapsed/i/60:.1f} 分")
    ranked = sorted(STAGE0_SCORE, key=lambda g: -STAGE0_SCORE[g])
    with open(stage0_file, "w") as f:
        f.write("rank\tgene\tscore\n")
        for rank, gene in enumerate(ranked, 1):
            f.write(f"{rank}\t{gene}\t{STAGE0_SCORE[gene]:.4f}\n")

STAGE0_RANK = {g: i for i, g in enumerate(sorted(STAGE0_SCORE, key=lambda g: -STAGE0_SCORE[g]), 1)}
STAGE0_PASS = sorted(STAGE0_SCORE, key=lambda g: -STAGE0_SCORE[g])[:STAGE0_KEEP]

print(f"\n段階0: {len(GENES)} → {len(STAGE0_PASS)}   Yes 側 {sum(v > 0 for v in STAGE0_SCORE.values())} 遺伝子")
print("上位 15:", ", ".join(f"{g}({STAGE0_SCORE[g]:+.1f})" for g in STAGE0_PASS[:15]))
for g in KNOWN_ANSWERS:
    if g in STAGE0_SCORE:
        print(f"  {g:<10} {STAGE0_RANK[g]:>6} 位  {STAGE0_SCORE[g]:+6.1f}  {'通過' if g in STAGE0_PASS else '⚠ 脱落'}")

## 9. 段階0.5 と 段階1 — 5 択＋「その他」で並べる

同じ処理を 2 回まわします（`CHOICE_ROUNDS` の 2 行）。1 回目（段階0.5）は粗く、2 回目（段階1）は詳しく並べます。

1 回分の手順:

1. **群を作る。** 候補をシャッフルした山札を「登場回数」の枚数だけつなげ、5 個ずつ区切ります。
   こうすると**全員がちょうど同じ回数**出題されます。同じ群に同じ遺伝子が 2 回入ったら、後ろの群と入れ替えます。
   （以前の勝ち残り方式では、早く落ちた遺伝子が 5 回、最後まで残った遺伝子が 35 回と、登場回数が 7 倍違いました。）
2. **出題する。** 選択肢の並びは問題ごとにシャッフルし、最後に「その他」を足します。
   モデルが答えの 1 文字目に `A`〜`F` のどれを出すかの確率を読み、選択肢ごとの確率に直します。
3. **Bradley-Terry で強さを出し**、上位 `残す数` 個を次へ送ります。表示する順位と次へ進む遺伝子は同じ基準です。

乱数の種を固定しているので、同じ設定なら毎回同じ群・同じ並びになります。
各回の出題は `stage0_5_matches.json` / `stage1_matches.json`、順位は `..._ranking.tsv` に保存します。

**注意:** 選択式は「5 つの中でいちばん」を選ばせるので、二値（段階0）より記述の書き方に強く左右されます。

In [ ]:
pool = list(STAGE0_PASS)
CHOICE_MATCHES = []          # 段階0.5・1 の全出題（最後の順位づけでも使う）
ROUND_RANK = {}              # 回ごとの順位（表示用）

# 選択式の見本（2 問）。本番と同じ形にする
choice_shots = ""
for disease, options, answer in CHOICE_EXAMPLES:
    lines = [f"{L}. {OPTION_TEXT[g]}" for L, g in zip(LETTERS, options)] + [f"{LETTERS[len(options)]}. {CHOICE_EXIT}"]
    choice_shots += (f"Disease: {disease}\nDisease mechanism:\n" + "".join(f"- {s}\n" for s in FEWSHOT_MECHANISM[disease])
                     + CHOICE_QUESTION + "\n" + "\n".join(lines) + f"\nAnswer: {LETTERS[options.index(answer)]}\n\n")

for round_name, keep, appearances, seed in CHOICE_ROUNDS:
    keep = min(keep, len(pool))
    size = min(GROUP_SIZE, len(pool))
    file_key = "stage" + round_name.replace("段階", "").replace(".", "_")      # stage0_5 / stage1
    matches_file = os.path.join(RUN_DIR, f"{file_key}_matches.json")
    print(f"\n===== {round_name}: {len(pool)} 遺伝子 × {appearances} 回 ÷ {size} 個 = {-(-len(pool)*appearances//size)} 問")

    if os.path.exists(matches_file):
        with open(matches_file) as f:
            matches = [(tuple(m["group"]), m["probs"]) for m in json.load(f)]
        print(f"  保存済みを読み込みました（{len(matches)} 問）")
    else:
        # ----- 1. 群を作る（全員ちょうど appearances 回）-----
        rng = random.Random(seed)
        sequence = []
        for _ in range(appearances):
            deck = list(pool)
            rng.shuffle(deck)
            sequence += deck
        for i in range(len(sequence)):                  # 同じ群に同じ遺伝子が入ったら、後ろの群と入れ替える
            start = i - i % size
            if sequence[i] in sequence[start:i]:
                for j in range(start + size, len(sequence)):
                    j_start = j - j % size
                    if sequence[j] not in sequence[start:start + size] and sequence[i] not in sequence[j_start:j_start + size]:
                        sequence[i], sequence[j] = sequence[j], sequence[i]
                        break
        groups = [sequence[i:i + size] for i in range(0, len(sequence), size)]

        # ----- 2. 出題する -----
        random.seed(seed)                               # 選択肢の並べ替えも固定する
        matches, exits, started = [], 0, time.time()
        for n, group in enumerate(groups, 1):
            options = list(group)
            random.shuffle(options)
            letter_of = {L: g for L, g in zip(LETTERS, options + [NONE])}
            lines = [f"{L}. {OPTION_TEXT[g]}" for L, g in zip(LETTERS, options)] + [f"{LETTERS[len(options)]}. {CHOICE_EXIT}"]
            prompt = ("Answer with a single letter only.\n\n" + choice_shots + mechanism_block
                      + CHOICE_QUESTION + "\n" + "\n".join(lines) + "\nAnswer:")
            logprobs = option_logprobs(prompt, [" " + L for L in letter_of])
            seen = {L: logprobs[L] for L in letter_of if L in logprobs}
            if not seen:
                print(f"  問 {n}: 選択肢の文字が返りませんでした（飛ばします）")
                continue
            floor = min(seen.values()) - 10.0               # 上位 TOP_K に出なかった文字は、いちばん低い値よりさらに下に置く
            top = max(seen.values())
            weights = {L: math.exp(seen.get(L, floor) - top) for L in letter_of}
            total = sum(weights.values())
            probs = {letter_of[L]: w / total for L, w in weights.items()}   # 選択肢ごとの確率（合計 1）
            exits += max(probs, key=probs.get) == NONE
            matches.append((tuple(group) + (NONE,), probs))
            if n == 10 or n % 200 == 0 or n == len(groups):
                elapsed = time.time() - started
                print(f"  {n}/{len(groups)} 問  経過 {elapsed/60:.1f} 分  残り {(len(groups)-n)*elapsed/n/60:.1f} 分")
        print(f"  「その他」が 1 位になった問: {exits}/{len(matches)}")
        with open(matches_file, "w") as f:
            json.dump([{"group": list(g), "probs": {k: round(v, 6) for k, v in p.items()}} for g, p in matches], f)

    # ----- 3. 強さを出して、上位 keep 個を残す -----
    strength = bradley_terry(matches)
    ranked = sorted(pool, key=lambda g: -strength.get(g, -99))
    ROUND_RANK[round_name] = {g: i for i, g in enumerate(ranked, 1)}
    with open(os.path.join(RUN_DIR, f"{file_key}_ranking.tsv"), "w") as f:
        f.write("rank\tgene\telo\tabove_none\tstage0_rank\n")
        for i, g in enumerate(ranked, 1):
            f.write(f"{i}\t{g}\t{to_elo(strength[g]):.1f}\t{strength[g] > strength[NONE]}\t{STAGE0_RANK[g]}\n")
    print(f"  「その他」の線（NONE）: elo {to_elo(strength[NONE]):.0f}  → 線より上 {sum(strength[g] > strength[NONE] for g in pool)} 遺伝子")
    print("  上位 15:", ", ".join(f"{g}({to_elo(strength[g]):.0f})" for g in ranked[:15]))
    for g in KNOWN_ANSWERS:
        if g in ROUND_RANK[round_name]:
            r = ROUND_RANK[round_name][g]
            print(f"  {g:<10} {r:>5} 位  {'通過' if r <= keep else '脱落'}")

    CHOICE_MATCHES += matches
    pool = ranked[:keep]

FINALISTS = pool
print(f"\n段階2 へ: {len(FINALISTS)} 遺伝子")

## 10. 段階2 — 残った遺伝子の総当たり

残った遺伝子のすべての組を、2 つ＋「どちらも関係ない」の 3 択で聞きます。

- **A/B を入れ替えて 2 回聞きます**（`BOTH_WAYS`）。モデルには「先に書いた方を選びやすい」癖があるので、両方向で打ち消します。
- 聞く回数は `n × (n − 1)` 回です。100 遺伝子なら 9,900 回、1 回 0.6〜0.9 秒で約 2 時間かかります。
- **1 回ごとに `stage2_matches.jsonl` に追記します。** 途中で止めても、`RESUME_DIR` を指定して実行し直せば続きから始まります。
- メカニズムを書かずに疾患名だけで聞くと、この形式はほぼ「いつも A を選ぶ」だけになりました。メカニズムの記述があって初めて比べられます。

In [ ]:
# 1 対 1 の見本（2 問）
pair_shots = ""
for disease, options, answer in PAIR_EXAMPLES:
    lines = [f"{L}. {OPTION_TEXT[g]}" for L, g in zip(LETTERS, options)] + [f"C. {PAIR_EXIT}"]
    pair_shots += (f"Disease: {disease}\nDisease mechanism:\n" + "".join(f"- {s}\n" for s in FEWSHOT_MECHANISM[disease])
                   + CHOICE_QUESTION + "\n" + "\n".join(lines) + f"\nAnswer: {LETTERS[options.index(answer)]}\n\n")

# 聞く組の一覧（A/B の順番つき）
orders = []
for i, a in enumerate(FINALISTS):
    for b in FINALISTS[i + 1:]:
        orders.append((a, b))
        if BOTH_WAYS:
            orders.append((b, a))

# 保存済みの対戦を読み込む（途中から再開するため）
pair_file = os.path.join(RUN_DIR, "stage2_matches.jsonl")
PAIR_MATCHES, done = [], set()
if os.path.exists(pair_file):
    with open(pair_file) as f:
        for line in f:
            m = json.loads(line)
            done.add(tuple(m["order"]))
            PAIR_MATCHES.append((tuple(m["group"]), m["probs"]))
todo = [o for o in orders if o not in done]
print(f"総当たり: {len(FINALISTS)} 遺伝子 → {len(orders)} 回（済み {len(done)}、残り {len(todo)}）")

started = time.time()
with open(pair_file, "a") as out:
    for n, (a, b) in enumerate(todo, 1):
        prompt = ("Answer with a single letter only.\n\n" + pair_shots + mechanism_block + CHOICE_QUESTION + "\n"
                  + f"A. {OPTION_TEXT[a]}\nB. {OPTION_TEXT[b]}\nC. {PAIR_EXIT}\nAnswer:")
        logprobs = option_logprobs(prompt, [" A", " B", " C"])
        letter_of = {"A": a, "B": b, "C": NONE}
        seen = {L: logprobs[L] for L in letter_of if L in logprobs}
        if not seen:
            continue                                        # 保存しないので、再開時に聞き直される
        floor, top = min(seen.values()) - 10.0, max(seen.values())
        weights = {L: math.exp(seen.get(L, floor) - top) for L in letter_of}
        total = sum(weights.values())
        probs = {letter_of[L]: round(w / total, 6) for L, w in weights.items()}
        PAIR_MATCHES.append(((a, b, NONE), probs))
        out.write(json.dumps({"order": [a, b], "group": [a, b, NONE], "probs": probs}) + "\n")
        out.flush()
        if n == 10 or n % 500 == 0 or n == len(todo):
            elapsed = time.time() - started
            print(f"  {n}/{len(todo)}  経過 {elapsed/60:.1f} 分  残り {(len(todo)-n)*elapsed/n/60:.1f} 分")

first_wins = sum(1 for g, p in PAIR_MATCHES if max(p, key=p.get) == g[0])
print(f"A 側が選ばれた割合: {first_wins}/{len(PAIR_MATCHES)}（半分前後なら位置の偏りは小さい）")

## 11. 最終順位

段階0.5・1 の選択式と段階2 の総当たりを**まとめて 1 回の Bradley-Terry に入れて**、段階2 に進んだ遺伝子を並べます。
どちらも「群と、選択肢ごとの確率」という同じ形なので、そのまま合わせられます。

表の見方:

- `elo`: 強さ。差 400 で「10 倍選ばれやすい」。
- `関係あり`: 仮想の遺伝子 `NONE`（「その他」「どちらも関係ない」）より強いかどうか。遺伝子数が多いと、ほぼ全員が「関係あり」になります。
- `段階0 / 段階0.5 / 段階1`: 各段階での順位。

結果は `final_ranking.tsv` に保存します。

In [ ]:
final_strength = bradley_terry(CHOICE_MATCHES + PAIR_MATCHES)
FINAL = sorted(FINALISTS, key=lambda g: -final_strength[g])
none_line = final_strength[NONE]

with open(os.path.join(RUN_DIR, "final_ranking.tsv"), "w") as f:
    f.write("rank\tgene\telo\trelevant\tstage0_rank\t" + "".join(f"{name}_rank\t" for name in ROUND_RANK) + "protein\n")
    for i, g in enumerate(FINAL, 1):
        f.write(f"{i}\t{g}\t{to_elo(final_strength[g]):.1f}\t{final_strength[g] > none_line}\t{STAGE0_RANK[g]}\t"
                + "".join(f"{ROUND_RANK[name][g]}\t" for name in ROUND_RANK) + f"{PROTEIN.get(g, '')}\n")

print(f"「関係ない」の線（NONE）: elo {to_elo(none_line):.0f}  → 関係あり {sum(final_strength[g] > none_line for g in FINAL)} / {len(FINAL)}\n")
print(f"{'順位':>4}  {'遺伝子':<10} {'elo':>6}  {'関係あり':<6} {'段階0':>6}" + "".join(f" {name:>7}" for name in ROUND_RANK) + "  蛋白質名")
for i, g in enumerate(FINAL, 1):
    print(f"{i:>4}  {g:<10} {to_elo(final_strength[g]):>6.0f}  {'○' if final_strength[g] > none_line else '×':<6} {STAGE0_RANK[g]:>6}"
          + "".join(f" {ROUND_RANK[name][g]:>7}" for name in ROUND_RANK) + f"  {PROTEIN.get(g, '')[:50]}")

print("\n答え合わせ:")
for g in KNOWN_ANSWERS:
    if g in FINAL:
        print(f"  {g:<10} 最終 {FINAL.index(g) + 1} 位 / {len(FINAL)}")
    elif any(g in ROUND_RANK[name] for name in ROUND_RANK):
        last = [name for name in ROUND_RANK if g in ROUND_RANK[name]][-1]      # 最後に出題された回で落ちた
        print(f"  {g:<10} {last} で脱落（{ROUND_RANK[last][g]} 位）")
    elif g in STAGE0_RANK:
        print(f"  {g:<10} 段階0 で脱落（{STAGE0_RANK[g]} 位）")
print("\n保存先:", RUN_DIR, sorted(os.listdir(RUN_DIR)))

## 12. 結果の読み方

- **上位は「この記述に合う遺伝子」です。** 別の妥当な記述（たとえば陰性症状に焦点を当てたもの、ドパミン仮説を軸にしたもの）で
  回すと、上位は大きく入れ替わります。大事な判断には、2〜3 通りの記述で回して、どの記述でも上位に来る遺伝子を見てください。
- **下位や脱落は「関係ない」の証拠ではありません。** モデルが知らないつながり（例: 承認薬の標的 CHRM4 と統合失調症）は拾えません。
  既知の標的の確認には使えますが、新しい標的を探す一次スクリーニングとしては過信しないでください。
- **同じ系統の遺伝子はまとめて上位に来やすい**です（グルタミン酸受容体の各サブユニットなど）。系統内の区別は苦手です。
- **温度 0 で、群と並びも乱数の種で固定**しているので、同じ設定なら同じ結果が返ります。再現性を確かめるには `CHOICE_ROUNDS` の種を変えてください。